# Adaptive Moving Average Window Evaluation

This notebook provides a comprehensive statistical analysis and evaluation comparing moving average smoothing models against naive persistence baselines across noisy synthetic time series.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import urllib.request
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-0ece95-adaptive-smoothing-and-persistence-trade/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded data successfully. Dataset keys:", list(data.keys()))

### Config Cell
Define evaluation configuration parameters.

In [ ]:
# Tunable parameters for the evaluation run
WINDOW_SIZE = 3
MAX_SAMPLES = 100

### Processing & Metrics Calculation
Extract predictions and actual values, compute MSE, RMSE, MAE, paired t-tests, and Wilcoxon signed-rank tests.

In [ ]:
all_examples = []
for ds in data.get("datasets", []):
    all_examples.extend(ds.get("examples", []))

all_examples = all_examples[:MAX_SAMPLES]
print(f"Processing {len(all_examples)} examples.")

actuals = []
ma_preds = []
naive_preds = []

for ex in all_examples:
    act = float(ex["output"])
    ma_p = float(ex["predict_moving_average"])
    naive_p = float(ex["predict_naive"])
    
    actuals.append(act)
    ma_preds.append(ma_p)
    naive_preds.append(naive_p)

actuals = np.array(actuals)
ma_preds = np.array(ma_preds)
naive_preds = np.array(naive_preds)

ma_errors = (actuals - ma_preds) ** 2
naive_errors = (actuals - naive_preds) ** 2

ma_mse = float(np.mean(ma_errors))
naive_mse = float(np.mean(naive_errors))
ma_rmse = float(np.sqrt(ma_mse))
naive_rmse = float(np.sqrt(naive_mse))
ma_mae = float(np.mean(np.abs(actuals - ma_preds)))
naive_mae = float(np.mean(np.abs(actuals - naive_preds)))

t_stat, p_value = stats.ttest_rel(naive_errors, ma_errors)
try:
    wilcoxon_stat, wilcoxon_p = stats.wilcoxon(naive_errors - ma_errors)
except Exception:
    wilcoxon_stat, wilcoxon_p = 0.0, 1.0

metrics_agg = {
    "moving_average_mse": ma_mse,
    "moving_average_rmse": ma_rmse,
    "moving_average_mae": ma_mae,
    "naive_persistence_mse": naive_mse,
    "naive_persistence_rmse": naive_rmse,
    "naive_persistence_mae": naive_mae,
    "mse_reduction": naive_mse - ma_mse,
    "percentage_improvement": float((naive_mse - ma_mse) / naive_mse * 100),
    "paired_t_stat": float(t_stat),
    "paired_t_p_value": float(p_value),
    "wilcoxon_stat": float(wilcoxon_stat),
    "wilcoxon_p_value": float(wilcoxon_p)
}

print("Aggregated Metrics:")
for k, v in metrics_agg.items():
    print(f"  {k}: {v:.6f}")

### Results & Visualization
Display summary metrics table and plot error comparisons between Moving Average and Naive Persistence models.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

models = ['Moving Average', 'Naive Persistence']
mse_values = [metrics_agg['moving_average_mse'], metrics_agg['naive_persistence_mse']]
rmse_values = [metrics_agg['moving_average_rmse'], metrics_agg['naive_persistence_rmse']]
mae_values = [metrics_agg['moving_average_mae'], metrics_agg['naive_persistence_mae']]

x = np.arange(len(models))
width = 0.25

ax.bar(x - width, mse_values, width, label='MSE', color='skyblue')
ax.bar(x, rmse_values, width, label='RMSE', color='salmon')
ax.bar(x + width, mae_values, width, label='MAE', color='lightgreen')

ax.set_ylabel('Error Value')
ax.set_title('Error Metrics Comparison: Moving Average vs Naive Persistence')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()

plt.tight_layout()
plt.show()